# Clinical NLP Training on Kaggle

Notebook này train XLM-R NER bằng annotation thật, reload checkpoint vừa train,
chạy entity linking/assertion và tạo `/kaggle/working/output.zip`.

Trước khi Run All: attach Dataset có input + annotation, bật GPU, và bật
Internet nếu không attach sẵn code/model. Xem `KAGGLE_RUNBOOK.md`.

## 1. Runtime and repository bootstrap

In [1]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

# [CRITICAL KAGGLE PATCH] Prevent TensorFlow and JAX from pre-allocating 100% of GPU memory
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['JAX_PLATFORM_NAME'] = 'cpu'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'


IS_KAGGLE = Path("/kaggle/input").is_dir()
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

GITHUB_REPO_URL = "https://github.com/takumi612/AI-Race-Viettel.git"
GITHUB_BRANCH = "main"
PROJECT_ROOT_OVERRIDE = ""
MODEL_NAME_OR_PATH_OVERRIDE = ""
INPUT_SOURCE_OVERRIDE = ""
TRAIN_SOURCE_OVERRIDE = ""
INSTALL_MISSING_DEPENDENCIES = True
FAST_DEV_RUN = True
REQUIRE_TRAINING_DATA = True
REQUIRE_GPU = True

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

def _is_project(path: Path) -> bool:
    return (path / "clinical_nlp_lab").is_dir() and (path / "artifacts/config.json").is_file()

project_candidates = []
if PROJECT_ROOT_OVERRIDE.strip():
    project_candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())
project_candidates.append(Path.cwd() / "Code_E_Platform")
project_candidates.append(Path.cwd())
if IS_KAGGLE:
    for marker in KAGGLE_INPUT_ROOT.rglob("clinical_nlp_lab"):
        if marker.is_dir():
            project_candidates.extend([marker.parent, marker.parent.parent / "Code_E_Platform"])

PROJECT_ROOT = next((path.resolve() for path in project_candidates if _is_project(path)), None)
clone_dir = KAGGLE_WORKING_ROOT / "AI-Race-Viettel"
if PROJECT_ROOT is None and IS_KAGGLE:
    clone_candidates = [clone_dir / "Code_E_Platform", clone_dir]
    if clone_dir.exists() and not any(_is_project(path) for path in clone_candidates):
        raise RuntimeError(f"Clone destination exists but is not a valid project: {clone_dir}")
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(clone_dir)],
            check=True,
        )
    PROJECT_ROOT = next((path.resolve() for path in clone_candidates if _is_project(path)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project code not found. Enable Internet for Git clone or attach a code Dataset and set PROJECT_ROOT_OVERRIDE."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_imports = {
    "torch": "torch",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bm25s": "bm25s",
    "faiss-cpu": "faiss",
    "sentence-transformers": "sentence_transformers"
}
missing = [package for package, module in required_imports.items() if importlib.util.find_spec(module) is None]
if IS_KAGGLE and missing and INSTALL_MISSING_DEPENDENCIES:
    requirements = PROJECT_ROOT / "requirements-kaggle.txt"
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
        importlib.invalidate_caches()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f"Could not install {missing}. Enable Internet or attach an environment/model Dataset."
        ) from exc

missing_after = [package for package, module in required_imports.items() if importlib.util.find_spec(module) is None]
DEPENDENCIES_READY = not missing_after
if IS_KAGGLE and missing_after:
    raise RuntimeError(f"Missing training dependencies after setup: {missing_after}")

print()
print("="*55)
print("📌 STEP 1: RUNTIME AND ENVIRONMENT BOOTSTRAP LOG")
print("="*55)
print(f"Is Kaggle Environment : {IS_KAGGLE}")
print(f"Project Root Directory: {PROJECT_ROOT}")
print(f"Python Version       : {sys.version.split()[0]}")
try:
    import torch
    print(f"PyTorch Version      : {torch.__version__}")
    print(f"CUDA Available       : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
        print(f"GPU Total VRAM       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
except Exception:
    pass
print(f"Dependencies Ready   : {DEPENDENCIES_READY}")
if missing_after:
    print(f"Missing Dependencies : {missing_after}")
print("="*55)
print()

Cloning into '/kaggle/working/AI-Race-Viettel'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.8 MB/s eta 0:00:00

📌 STEP 1: RUNTIME AND ENVIRONMENT BOOTSTRAP LOG
Is Kaggle Environment : True
Project Root Directory: /kaggle/working/AI-Race-Viettel/Code_E_Platform
Python Version       : 3.12.13
PyTorch Version      : 2.10.0+cu128
CUDA Available       : True
GPU Device Name      : Tesla T4
GPU Total VRAM       : 14.56 GB
Dependencies Ready   : True



## 2. Discover attached input and annotations

In [2]:
def _has_text_files(path: Path) -> bool:
    return path.is_dir() and any(path.glob("*.txt"))

def _is_archive_path(path: Path, search_root: Path) -> bool:
    try:
        relative_parts = path.relative_to(search_root).parts
    except ValueError:
        relative_parts = path.parts
    return bool({"test", "archive", "runtime_evidence"} & {part.lower() for part in relative_parts})

def _has_training_layout(path: Path) -> bool:
    if not path.is_dir():
        return False
    text_stems = {item.stem for item in path.glob("*.txt")}
    json_stems = {item.stem for item in path.glob("*.json")}
    direct_pairs = bool(text_stems & json_stems)
    split_pairs = _has_text_files(path / "input") and (path / "gt").is_dir() and any((path / "gt").glob("*.json"))
    return direct_pairs or split_pairs

search_roots = [PROJECT_ROOT]
if IS_KAGGLE:
    search_roots = [path for path in KAGGLE_INPUT_ROOT.iterdir() if path.is_dir()] + search_roots
else:
    search_roots.extend(ancestor for ancestor in PROJECT_ROOT.parents if (ancestor / "input.zip").is_file())

input_candidates = []
if INPUT_SOURCE_OVERRIDE.strip():
    input_candidates.append(Path(INPUT_SOURCE_OVERRIDE).expanduser())
for root in search_roots:
    input_candidates.extend(path for path in root.rglob("input.zip") if not _is_archive_path(path, root))
    input_candidates.extend(
        path for path in root.rglob("input")
        if _has_text_files(path)
        and path.parent.name.lower() not in {"synthetic_train_v1", "train"}
        and not _is_archive_path(path, root)
    )
INPUT_SOURCE = next((path.resolve() for path in input_candidates if path.is_file() or _has_text_files(path)), None)
if INPUT_SOURCE is None:
    raise FileNotFoundError(
        "Real inference input not found. Attach a Dataset containing input.zip or input/<id>.txt."
    )

train_candidates = []
if TRAIN_SOURCE_OVERRIDE.strip():
    train_candidates.append(Path(TRAIN_SOURCE_OVERRIDE).expanduser())
for root in search_roots:
    train_candidates.extend(path for path in root.rglob("train") if not _is_archive_path(path, root))
    train_candidates.extend(path for path in root.rglob("synthetic_train_v1") if not _is_archive_path(path, root))

# Local-only fixture lets the generated notebook be executed as a smoke test.
if not IS_KAGGLE:
    local_fixture_candidates = [
        ancestor / "test/archive/tests/fixtures/paired_annotations"
        for ancestor in (PROJECT_ROOT, *PROJECT_ROOT.parents)
    ]
    local_fixture = next(
        (path for path in local_fixture_candidates if _has_training_layout(path)),
        None,
    )
    if local_fixture is not None:
        train_candidates.insert(0, local_fixture)

TRAIN_SOURCE = next((path.resolve() for path in train_candidates if _has_training_layout(path)), None)
if TRAIN_SOURCE is None and REQUIRE_TRAINING_DATA:
    raise FileNotFoundError(
        "No annotated training data found. Attach train/*.txt + *.json or synthetic_train_v1/input + gt."
    )

if IS_KAGGLE:
    RUN_ROOT = KAGGLE_WORKING_ROOT
else:
    RUN_ROOT = PROJECT_ROOT / "test/runtime_evidence/kaggle_notebook_simulation"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
TRAINING_ROOT = RUN_ROOT / "training_artifacts"
NER_MODEL_DIR = TRAINING_ROOT / "ner_model"
OUTPUT_DIR = RUN_ROOT / "output"
DIAGNOSTICS_DIR = RUN_ROOT / "diagnostics"
OUTPUT_ZIP = RUN_ROOT / "output.zip"
TRAINED_ARTIFACTS_ZIP = RUN_ROOT / "trained_ner_artifacts.zip"

print()
print("="*55)
print("📂 STEP 2: DATASET DISCOVERY LOG")
print("="*55)
print(f"Inference Input Source: {INPUT_SOURCE}")
print(f"Training Data Source  : {TRAIN_SOURCE}")
print(f"Run Working Directory : {RUN_ROOT}")
print(f"Output Zip Target     : {OUTPUT_ZIP}")
print("="*55)
print()


📂 STEP 2: DATASET DISCOVERY LOG
Inference Input Source: /kaggle/input/datasets/bacreal3k/ai-race-clinical-data/ai-race-clinical-data/input/input
Training Data Source  : /kaggle/input/datasets/bacreal3k/ai-race-clinical-data/ai-race-clinical-data/synthetic_train_v1
Run Working Directory : /kaggle/working
Output Zip Target     : /kaggle/working/output.zip



## 3. Validate data, split by document, and check GPU

In [3]:
from clinical_nlp_lab.config import load_config, set_reproducible_seed
from clinical_nlp_lab.data import (
    document_train_validation_split,
    load_annotated_documents,
    load_input_documents,
    validate_documents,
)
from clinical_nlp_lab.schema import write_json

CONFIG = load_config(PROJECT_ROOT / "artifacts/config.json")
SEED_STATUS = set_reproducible_seed(int(CONFIG["seed"]))
INPUT_DOCUMENTS = load_input_documents(INPUT_SOURCE)
ANNOTATED_DOCUMENTS = load_annotated_documents(TRAIN_SOURCE) if TRAIN_SOURCE else []
ANNOTATION_REPORT = validate_documents(ANNOTATED_DOCUMENTS)
if not ANNOTATION_REPORT["is_valid"]:
    raise ValueError(f"Training annotation validation failed: {ANNOTATION_REPORT['errors'][:10]}")
if REQUIRE_TRAINING_DATA and not ANNOTATED_DOCUMENTS:
    raise ValueError("Training is required but no annotated documents were loaded")

TRAIN_DOCUMENTS, VALIDATION_DOCUMENTS = document_train_validation_split(
    ANNOTATED_DOCUMENTS,
    float(CONFIG["validation_fraction"]),
    int(CONFIG["seed"]),
)
if FAST_DEV_RUN:
    TRAIN_DOCUMENTS = TRAIN_DOCUMENTS[: min(16, len(TRAIN_DOCUMENTS))]
    VALIDATION_DOCUMENTS = VALIDATION_DOCUMENTS[: min(4, len(VALIDATION_DOCUMENTS))]

GPU_STATUS = {"available": False, "name": None}
if DEPENDENCIES_READY:
    import torch
    GPU_STATUS = {
        "available": torch.cuda.is_available(),
        "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
if IS_KAGGLE and REQUIRE_GPU and not GPU_STATUS["available"]:
    raise RuntimeError("GPU is required. Open Kaggle Settings and select a GPU accelerator.")

print()
print("="*55)
print("📊 STEP 3: DATASET STATISTICS & VALIDATION LOG")
print("="*55)
print(f"Input Inference Docs  : {len(INPUT_DOCUMENTS)}")
print(f"Annotated Train Docs  : {len(ANNOTATED_DOCUMENTS)}")
print(f"Train Split Docs      : {len(TRAIN_DOCUMENTS)}")
print(f"Validation Split Docs : {len(VALIDATION_DOCUMENTS)}")
print(f"Total GT Entities     : {ANNOTATION_REPORT.get('entity_count', 0)}")
if ANNOTATION_REPORT.get("by_type"):
    print("Entity Type Breakdown :")
    for etype, count in ANNOTATION_REPORT["by_type"].items():
        print(f"  - {etype:<15}: {count}")
print(f"GPU Accelerator Status: {GPU_STATUS}")
print(f"Reproducible Seed     : {SEED_STATUS.get('seed')}")
print("="*55)
print()


📊 STEP 3: DATASET STATISTICS & VALIDATION LOG
Input Inference Docs  : 100
Annotated Train Docs  : 200
Train Split Docs      : 16
Validation Split Docs : 4
Total GT Entities     : 17004
GPU Accelerator Status: {'available': True, 'name': 'Tesla T4'}
Reproducible Seed     : None



## 4. Training configuration

In [4]:
def _looks_like_xlmr_model(path: Path) -> bool:
    config_path = path / "config.json"
    if not config_path.is_file():
        return False
    try:
        payload = json.loads(config_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return payload.get("model_type") in {"xlm-roberta", "roberta"} and (
        (path / "tokenizer.json").is_file()
        or (path / "sentencepiece.bpe.model").is_file()
    )

attached_models = []
if IS_KAGGLE:
    attached_models = [path.parent for path in KAGGLE_INPUT_ROOT.rglob("config.json") if _looks_like_xlmr_model(path.parent)]

if MODEL_NAME_OR_PATH_OVERRIDE.strip():
    MODEL_SOURCE = MODEL_NAME_OR_PATH_OVERRIDE
elif attached_models:
    MODEL_SOURCE = str(attached_models[0])
else:
    MODEL_SOURCE = str(CONFIG["ner_model_name"])

NER_EPOCHS = 1 if FAST_DEV_RUN else int(CONFIG["ner_epochs"])
TRAIN_BATCH_SIZE = 2 if FAST_DEV_RUN else int(CONFIG["batch_size"])
LEARNING_RATE = float(CONFIG["learning_rate"])

print()
print("="*55)
print("⚙️ STEP 4: TRAINING HYPERPARAMETERS & CONFIGURATION")
print("="*55)
print(f"Base Model Source     : {MODEL_SOURCE}")
print(f"Training Epochs       : {NER_EPOCHS}")
print(f"Batch Size (per GPU)  : {TRAIN_BATCH_SIZE}")
print(f"Learning Rate         : {LEARNING_RATE}")
print(f"Max Sequence Length   : {CONFIG.get('max_length')}")
print(f"Window Stride         : {CONFIG.get('stride')}")
print(f"Fast Dev Mode Run     : {FAST_DEV_RUN}")
print("="*55)
print()


⚙️ STEP 4: TRAINING HYPERPARAMETERS & CONFIGURATION
Base Model Source     : xlm-roberta-base
Training Epochs       : 1
Batch Size (per GPU)  : 2
Learning Rate         : 0.0002
Max Sequence Length   : 512
Window Stride         : 128
Fast Dev Mode Run     : True



## 4.5 Check GPU Memory (Diagnostic)

Chạy cell này để đảm bảo VRAM hoàn toàn trống trước khi train.
Nếu thấy `Memory-Usage` lớn hơn 1GB dù chưa train, bạn đang bị kẹt tiến trình ẩn (Zombie Process). Hãy bấm nút **Power Off** ở góc phải Kaggle để tắt hẳn máy ảo và bật lại.

In [5]:
import subprocess
import sys
print("="*55)
print("🔍 GPU MEMORY DIAGNOSTIC")
print("="*55)
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except Exception as e:
    print(f"Không thể chạy nvidia-smi: {e}")
print("="*55)

🔍 GPU MEMORY DIAGNOSTIC
Tue Jul 21 03:45:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             27W /   70W |     113MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------

## 5. Train XLM-R token classifier

In [6]:
import gc
import inspect
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
from clinical_nlp_lab.training import build_bio_label_map, prepare_token_classification_features

if DEPENDENCIES_READY and TRAIN_DOCUMENTS:
    if (NER_MODEL_DIR / "model.safetensors").exists() or (NER_MODEL_DIR / "pytorch_model.bin").exists():
        print(f"[SKIP] Found existing NER weights at {NER_MODEL_DIR}, skipping training.")
        NER_TRAINING_RESULT = {"trained": False, "reason": "checkpoint_exists", "model_dir": str(NER_MODEL_DIR)}
    else:
        # ---- BẮT ĐẦU INLINE TRAINING LÊN IPYNB ĐỂ DỄ QUẢN LÝ BỘ NHỚ ----
        print("Bắt đầu chuẩn bị dữ liệu train...")
        entity_types = {entity.type for document in TRAIN_DOCUMENTS for entity in document.entities}
        label_to_id, id_to_label = build_bio_label_map(entity_types)
        
        tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, use_fast=True)
        train_features = prepare_token_classification_features(
            TRAIN_DOCUMENTS, tokenizer, label_to_id, int(CONFIG["max_length"]), int(CONFIG["stride"])
        )
        validation_features = prepare_token_classification_features(
            VALIDATION_DOCUMENTS, tokenizer, label_to_id, int(CONFIG["max_length"]), int(CONFIG["stride"])
        )

        class FeatureDataset(Dataset):
            def __init__(self, features): self.features = features
            def __len__(self): return len(self.features)
            def __getitem__(self, index):
                return {k: v for k, v in self.features[index].items() if k not in {"offset_mapping", "document_id"}}

        print("Tải model lên GPU...")
        model = AutoModelForTokenClassification.from_pretrained(
            MODEL_SOURCE, num_labels=len(label_to_id), label2id=label_to_id, id2label=id_to_label
        )
        
        output_path = Path(NER_MODEL_DIR)
        output_path.mkdir(parents=True, exist_ok=True)
        
        # Ép batch_size nhỏ xuống để chống OOM
        SAFE_BATCH_SIZE = 2
        grad_accum_steps = int(CONFIG.get("gradient_accumulation_steps", 4))
        
        training_kwargs = {
            "output_dir": str(output_path),
            "learning_rate": LEARNING_RATE,
            "num_train_epochs": NER_EPOCHS,
            "per_device_train_batch_size": SAFE_BATCH_SIZE,
            "per_device_eval_batch_size": SAFE_BATCH_SIZE,
            "weight_decay": 0.01,
            "warmup_ratio": 0.1,
            "save_strategy": "epoch" if validation_features else "no",
            "save_total_limit": 1,
            "load_best_model_at_end": bool(validation_features),
            "seed": int(CONFIG["seed"]),
            "report_to": [],
            "fp16": torch.cuda.is_available(),
            "gradient_accumulation_steps": grad_accum_steps,
            "gradient_checkpointing": True, # Tiết kiệm 40% VRAM
        }
        
        argument_parameters = inspect.signature(TrainingArguments.__init__).parameters
        evaluation_key = "eval_strategy" if "eval_strategy" in argument_parameters else "evaluation_strategy"
        training_kwargs[evaluation_key] = "epoch" if validation_features else "no"
        arguments = TrainingArguments(**training_kwargs)
        
        import subprocess
        trainer = Trainer(
            model=model,
            args=arguments,
            train_dataset=FeatureDataset(train_features),
            eval_dataset=FeatureDataset(validation_features) if validation_features else None,
            data_collator=DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=8),
        )
        trainer.processing_class = tokenizer
        
        print(f"Bắt đầu gọi Trainer.train() với batch={SAFE_BATCH_SIZE}, grad_accum={grad_accum_steps}...")
        train_result = trainer.train()
        trainer.save_model(str(output_path))
        tokenizer.save_pretrained(str(output_path))
        
        metrics = train_result.metrics
        NER_TRAINING_RESULT = {
            "trained": True,
            "train_chunks": len(train_features),
            "validation_chunks": len(validation_features),
            "training_loss": metrics.get("train_loss", 0.0),
            "output_dir": str(output_path),
        }
        
        # Dọn dẹp VRAM ngay sau khi train xong!
        print("Train xong. Đang giải phóng bộ nhớ GPU...")
        del model
        del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    NER_TRAINING_RESULT = {"trained": False, "reason": "No training documents"}

TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
from clinical_nlp_lab.schema import write_json
write_json(TRAINING_ROOT / "training_result.json", NER_TRAINING_RESULT)


print('='*55)
print('🚀 STEP 5: XLM-R NER MODEL TRAINING EXECUTION')
print('='*55)
print(f'✅ Training Status     : COMPLETED')


Bắt đầu chuẩn bị dữ liệu train...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,No log,1.159588


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train xong. Đang giải phóng bộ nhớ GPU...
🚀 STEP 5: XLM-R NER MODEL TRAINING EXECUTION
✅ Training Status     : COMPLETED


## 6. Package the trained checkpoint

In [7]:
archive_base = TRAINED_ARTIFACTS_ZIP.with_suffix("")
created_archive = shutil.make_archive(str(archive_base), "zip", root_dir=TRAINING_ROOT)
assert Path(created_archive).is_file()
zip_size_mb = Path(created_archive).stat().st_size / (1024 * 1024)
print(f"📦 Checkpoint Packaged: {created_archive} ({zip_size_mb:.2f} MB)")

📦 Checkpoint Packaged: /kaggle/working/trained_ner_artifacts.zip (2167.14 MB)


## 6.5 Garbage collection

In [8]:
import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 GPU Memory Cleaned (empty_cache & gc.collect).")

🧹 GPU Memory Cleaned (empty_cache & gc.collect).


## 7. Inference with the newly trained NER model

In [9]:
# Cài vllm + autoawq lazy (chỉ khi cần Inference, không ảnh hưởng Training)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm', 'autoawq'], check=False)

from clinical_nlp_lab.pipeline import run_inference

ACTIVE_NER_MODEL = NER_MODEL_DIR if (NER_TRAINING_RESULT.get("trained") or NER_TRAINING_RESULT.get("reason") == "checkpoint_exists") else None
INFERENCE_SUMMARY = run_inference(
    INPUT_SOURCE,
    OUTPUT_DIR,
    PROJECT_ROOT / "artifacts",
    create_zip=True,
    diagnostics_dir=DIAGNOSTICS_DIR,
    zip_path=OUTPUT_ZIP,
    ner_model_dir=ACTIVE_NER_MODEL,
    train_source=TRAIN_SOURCE,
)

print()
print("="*55)
print("⚡ STEP 7: INFERENCE PIPELINE EXECUTION LOG")
print("="*55)
print(f"Active NER Model      : {INFERENCE_SUMMARY.get('active_ner')}")
print(f"Documents Inferred    : {INFERENCE_SUMMARY.get('documents_processed')}")
print(f"Total Entities Output : {INFERENCE_SUMMARY.get('submission_entity_count')}")
if INFERENCE_SUMMARY.get("by_type"):
    print("Entity Type Breakdown :")
    for etype, count in INFERENCE_SUMMARY["by_type"].items():
        print(f"  - {etype:<15}: {count}")
print(f"Output Zip Path       : {INFERENCE_SUMMARY.get('zip_path')}")
print("="*55)
print()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.1/250.1 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.0/458.0 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.3.1 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but

[RE-INDEX] Enriched BM25+FAISS candidate records from train data: +348 ICD-10 aliases, +94 RxNorm aliases.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Split strings:   0%|          | 0/46849 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[LLM-SUBPROCESS] Running LLM Reranking & Assertion in subprocess...
[LLM-SUBPROCESS] WARNING: LLM subprocess failed. Continuing without LLM reranking/assertion.
Stdout: INFO 07-21 04:01:13 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
INFO 07-21 04:01:29 [model.py:619] Resolved architecture: Qwen2ForCausalLM
INFO 07-21 04:01:29 [model.py:1776] Using max model len 4096
INFO 07-21 04:01:30 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-21 04:01:31 [vllm.py:1042] Asynchronous scheduling is enabled.
WARNING 07-21 04:01:31 [vllm.py:1096] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 07-21 04:01:31 [vllm.py:1144] Inductor compilation was disabled by user settings, optimizations se

## 8. Validate submission and write run manifest

In [10]:
import zipfile
from clinical_nlp_lab.schema import validate_submission_payload

schema_errors = {}
for document in INPUT_DOCUMENTS:
    prediction_path = OUTPUT_DIR / f"{document.document_id}.json"
    prediction = json.loads(prediction_path.read_text(encoding="utf-8"))
    errors = validate_submission_payload(prediction, document.raw_text)
    if errors:
        schema_errors[document.document_id] = errors
if schema_errors:
    raise ValueError(f"Submission schema errors: {list(schema_errors.items())[:3]}")

with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    zip_names = archive.namelist()
    assert len(zip_names) == len(INPUT_DOCUMENTS)
    assert all(name.startswith("output/") and not name.startswith("output/output/") for name in zip_names)
    assert archive.testzip() is None

RUN_MANIFEST = {
    "trained": bool(NER_TRAINING_RESULT.get("trained")),
    "active_ner": INFERENCE_SUMMARY.get("active_ner"),
    "train_documents": len(TRAIN_DOCUMENTS),
    "validation_documents": len(VALIDATION_DOCUMENTS),
    "input_documents": len(INPUT_DOCUMENTS),
    "submission_entities": INFERENCE_SUMMARY["submission_entity_count"],
    "schema_error_count": 0,
    "offset_error_count": INFERENCE_SUMMARY["offset_error_count"],
    "output_zip": str(OUTPUT_ZIP),
    "trained_artifacts_zip": str(TRAINED_ARTIFACTS_ZIP),
}
write_json(RUN_ROOT / "run_manifest.json", RUN_MANIFEST)
print(RUN_MANIFEST)

{'trained': True, 'active_ner': 'hybrid_transformer_and_dictionary', 'train_documents': 16, 'validation_documents': 4, 'input_documents': 100, 'submission_entities': 843, 'schema_error_count': 0, 'offset_error_count': 0, 'output_zip': '/kaggle/working/output.zip', 'trained_artifacts_zip': '/kaggle/working/trained_ner_artifacts.zip'}


## 9. Download results

Kaggle outputs:

- `/kaggle/working/output.zip`
- `/kaggle/working/trained_ner_artifacts.zip`
- `/kaggle/working/run_manifest.json`

Chọn **Save Version → Save & Run All**, sau đó tải file từ tab Output.